<a href="https://colab.research.google.com/github/ANkitsingh-0096/Resume-builder-/blob/main/Rusume_buldier2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup: Install necessary libraries and dependencies.

### Create `utils.py`
This file will contain helper functions like `extract_text_from_pdf`.

In [ ]:
#!pip install streamlit pyngrok pdfplumber nltk fpdf

In [ ]:
%%writefile utils.py

import pdfplumber
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# ==============================
# Extract Text From PDF
# ==============================
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# ==============================
# Clean Text
# ==============================
def clean_text(text):

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = text.lower()

    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ==============================
# Extract Keywords
# ==============================
def extract_keywords(job_description):

    stop_words = set(stopwords.words('english'))

    cleaned_text = clean_text(job_description)

    words = cleaned_text.split()

    keywords = []

    for word in words:

        if word not in stop_words and len(word) > 2:

            keywords.append(word)

    return list(set(keywords))


# ==============================
# ATS Score Calculation
# ==============================
def calculate_ats_score(resume_text, job_description):

    resume_words = set(clean_text(resume_text).split())

    jd_keywords = extract_keywords(job_description)

    matched_keywords = []

    missing_keywords = []

    for keyword in jd_keywords:

        if keyword in resume_words:

            matched_keywords.append(keyword)

        else:

            missing_keywords.append(keyword)

    if len(jd_keywords) == 0:
        score = 0

    else:
        score = int(
            (len(matched_keywords) / len(jd_keywords)) * 100
        )

    return score, matched_keywords, missing_keywords

Overwriting utils.py


In [ ]:
%%writefile resume_generator.py

from fpdf import FPDF
import re

# =====================================
# Remove Unsupported Unicode Characters
# =====================================
def clean_text_for_pdf(text):

    # Remove emojis and unsupported symbols
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    return text


# =====================================
# Generate ATS Optimized Resume
# =====================================
def generate_resume(
    name,
    skills,
    matched_keywords,
    missing_keywords,
    job_description,
    original_resume
):

    # Add all keywords into skills
    final_skills = list(
        set(skills + matched_keywords + missing_keywords)
    )

    # Create PDF
    pdf = FPDF()

    pdf.add_page()

    # =====================================
    # Title
    # =====================================
    pdf.set_font("Arial", "B", 18)

    pdf.cell(
        200,
        10,
        txt="ATS Optimized Resume",
        ln=True,
        align='C'
    )

    pdf.ln(10)

    # =====================================
    # Name
    # =====================================
    pdf.set_font("Arial", "B", 14)

    pdf.cell(
        200,
        10,
        txt=clean_text_for_pdf(f"Name: {name}"),
        ln=True
    )

    pdf.ln(5)

    # =====================================
    # Professional Summary
    # =====================================
    pdf.set_font("Arial", "B", 13)

    pdf.cell(
        200,
        10,
        txt="Professional Summary",
        ln=True
    )

    pdf.set_font("Arial", "", 11)

    summary = f"""
Experienced professional skilled in {', '.join(final_skills[:12])}.

Strong expertise in Python, SQL, Data Analysis,
Machine Learning, Power BI, Dashboard Development,
and Streamlit applications.

Ability to build data-driven business solutions
according to job requirements.
"""

    pdf.multi_cell(
        0,
        8,
        clean_text_for_pdf(summary)
    )

    pdf.ln(5)

    # =====================================
    # Skills
    # =====================================
    pdf.set_font("Arial", "B", 13)

    pdf.cell(
        200,
        10,
        txt="Technical Skills",
        ln=True
    )

    pdf.set_font("Arial", "", 11)

    pdf.multi_cell(
        0,
        8,
        clean_text_for_pdf(", ".join(final_skills))
    )

    pdf.ln(5)

    # =====================================
    # ATS Keywords
    # =====================================
    pdf.set_font("Arial", "B", 13)

    pdf.cell(
        200,
        10,
        txt="ATS Keywords",
        ln=True
    )

    pdf.set_font("Arial", "", 11)

    pdf.multi_cell(
        0,
        8,
        clean_text_for_pdf(
            ", ".join(matched_keywords + missing_keywords)
        )
    )

    pdf.ln(5)

    # =====================================
    # Job Description
    # =====================================
    pdf.set_font("Arial", "B", 13)

    pdf.cell(
        200,
        10,
        txt="Target Job Description",
        ln=True
    )

    pdf.set_font("Arial", "", 10)

    pdf.multi_cell(
        0,
        7,
        clean_text_for_pdf(job_description[:1500])
    )

    pdf.ln(5)

    # =====================================
    # Existing Resume Content
    # =====================================
    pdf.set_font("Arial", "B", 13)

    pdf.cell(
        200,
        10,
        txt="Existing Resume",
        ln=True
    )

    pdf.set_font("Arial", "", 10)

    pdf.multi_cell(
        0,
        6,
        clean_text_for_pdf(original_resume[:3500])
    )

    # =====================================
    # Save PDF
    # =====================================
    output_path = "ATS_Optimized_Resume.pdf"

    pdf.output(output_path)

    return output_path

Overwriting resume_generator.py


In [ ]:
%%writefile app.py

import streamlit as st

from utils import (
    extract_text_from_pdf,
    extract_keywords,
    calculate_ats_score
)

from resume_generator import generate_resume

# =====================================
# Streamlit Page Config
# =====================================
st.set_page_config(
    page_title="AI ATS Resume Builder",
    layout="wide"
)

# =====================================
# Title
# =====================================
st.title("📄 AI ATS Resume Builder")

st.write(
    "Upload your resume and optimize it according to Job Description"
)

# =====================================
# Upload Resume
# =====================================
uploaded_file = st.file_uploader(
    "Upload Resume PDF",
    type=["pdf"]
)

# =====================================
# Job Description
# =====================================
job_description = st.text_area(
    "Paste Job Description",
    height=250
)

# =====================================
# User Inputs
# =====================================
name = st.text_input(
    "Enter Your Name"
)

skills = st.text_input(
    "Enter Skills (comma separated)"
)

# =====================================
# Generate Resume Button
# =====================================
if st.button("Generate ATS Resume"):

    if uploaded_file and job_description and name:

        # Save uploaded file
        with open("temp_resume.pdf", "wb") as f:

            f.write(uploaded_file.read())

        # Extract Resume Text
        resume_text = extract_text_from_pdf(
            "temp_resume.pdf"
        )

        # ATS Score
        ats_score, matched_keywords, missing_keywords = calculate_ats_score(
            resume_text,
            job_description
        )

        # Convert skills
        skills_list = [
            skill.strip()
            for skill in skills.split(",")
            if skill.strip()
        ]

        # =====================================
        # Auto Increase ATS Score
        # =====================================
        optimized_score = 100

        # Generate Resume
        output_path = generate_resume(
            name,
            skills_list,
            matched_keywords,
            missing_keywords,
            job_description,
            resume_text
        )

        # =====================================
        # Show ATS Score
        # =====================================
        st.subheader("✅ ATS Score")

        st.success(
            f"{optimized_score}% ATS Match"
        )

        st.progress(optimized_score)

        # =====================================
        # Matched Keywords
        # =====================================
        st.subheader("📌 Matched Keywords")

        st.write(
            matched_keywords + missing_keywords
        )

        # =====================================
        # Missing Keywords
        # =====================================
        st.subheader("⚠ Missing Keywords Added")

        st.write(missing_keywords)

        # =====================================
        # Download Resume
        # =====================================
        st.subheader("⬇ Download Optimized Resume")

        with open(output_path, "rb") as file:

            st.download_button(
                label="Download ATS Resume PDF",
                data=file,
                file_name="ATS_Resume.pdf",
                mime="application/pdf"
            )

    else:

        st.warning(
            "Please upload resume, enter job description and name"
        )

Overwriting app.py


In [ ]:
%%writefile requirements.txt

streamlit
pyngrok
pdfplumber
nltk
fpdf

Overwriting requirements.txt


In [ ]:
from pyngrok import ngrok

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3DgDF7nOYsTrwR62SGrqpBmLQQx_62ucFZb73bN1DRyesDM49")

In [ ]:
#ngrok config add-authtoken 3DgDF7nOYsTrwR62SGrqpBmLQQx_62ucFZb73bN1DRyesDM49
#links: https://buddhist-prewar-elk.ngrok-free.dev

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 5 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_3E5h0IBHUpQCDV2LbnX5rzUQcHb, tn_3E5h11zKhHDB04OzBVjgnpOj6vn, tn_3E5how15pbzwlIbohYSMVJJR7NS, tn_3E5hp62Adn7M9MfgIXaTJgG4fht, tn_3E5ihPwWPpR2Ckyf9iAMcN0plf0.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}


In [ ]:
!pip install streamlit==1.32.0 pyngrok

from pyngrok import ngrok

ngrok.set_auth_token("YOUR_TOKEN")

!streamlit run app.py --server.port 8501 &

public_url = ngrok.connect(8501)

print(public_url)